# Evaluation and provenance channels — one program, either transport

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/book/14-eval-channels/eval-channels.ipynb)

Built from [`cookbook/book/chapters/14-eval-channels/eval-channels.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/14-eval-channels/eval-channels.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures. The chapter runs
# at `small` scale, over the committed fixtures, in minutes. SCALE = "full" runs
# it over the published data and real encoders instead: meant for a GPU, and the
# chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook==0.49.1", server + "==0.49.1"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

In [ ]:
import jammi_cookbook

**Recipe:** `eval_embeddings` (with cohorts) · `eval_per_query` ·
`eval_compare` · `eval_inference` (classification and NER) ·
`register_channel` · `add_channel_columns` · `list_channels` · `jammi.testing.LiveServer`
· **Theory:** retrieval and inference *evaluation as an engine verb* —
precision\@k / recall\@k / MRR / nDCG, accuracy / F1, entity-span
precision/recall — and the evidence-provenance channel as a typed, append-only,
tenant-scoped registry (Kleppmann 2017) · **Rail:** measurement (every
aggregate against its golden) + parity (the remote report equals the embedded
one, value for value) + tenancy (channel isolation, measured).

Every evaluation in the keystone ran on the embedded engine. The same verbs are
on a `jammi-server`, and a program written against a `Session` runs unchanged on
either. This chapter makes that a measured property rather than a claim: it
writes the evaluation once, as one function, runs it against an in-process
engine and against a live server, and asserts the two reports are the same.

In [ ]:
import tempfile
import uuid

import jammi
from jammi.testing import LiveServer
from jammi_cookbook import contracts, datasets, fixtures, keystone, scale

SCALE = scale.current()
TENANTS = (str(uuid.uuid4()), str(uuid.uuid4()))

## The evaluation, written once

Four evaluations and one provenance sequence, over a `Session` of either kind:

- **`eval_embeddings`** scores the keystone's raw embeddings on same-subject
  retrieval, with each query tagged by its **cohort** — the time-split era its
  paper belongs to — so the report can be sliced; **`eval_per_query`** reads the
  run's persisted per-query rows back;
- **`eval_compare`** scores the raw and graph-propagated tables side by side,
  with a paired significance test on the difference;
- **`eval_inference`** runs a model's forward pass and scores it:
  classification (accuracy / F1 against a label set) and NER (entity-span
  precision / recall against character-offset gold spans), on the cookbook's
  small fixture models;
- the **channel sequence** registers two provenance channels under one tenant,
  appends columns, and reads the registry from both tenants and from no tenant.

In [ ]:
def evaluate(db) -> dict:
    arxiv = datasets.arxiv(db, SCALE)
    raw = keystone.embed(db, arxiv, SCALE)
    propagated = keystone.propagate(db, arxiv, raw)
    golden = keystone.subject_golden(db, arxiv)
    era = {pid: name for name, ids in arxiv.split.items() for pid in ids}
    queries = db.sql(f"SELECT DISTINCT query_id FROM {golden}").column(0).to_pylist()

    retrieval = db.eval_embeddings(
        source=arxiv.papers, golden_source=golden, embedding_table=raw, k=10,
        cohorts={q: {"era": era[q]} for q in queries},
    )
    per_query = db.eval_per_query(retrieval["eval_run_id"])
    compared = db.eval_compare(
        embedding_tables=[raw, propagated], source=arxiv.papers, golden_source=golden, k=10,
    )

    db.add_source("corpus", url=fixtures.url("tiny_corpus.parquet"), format="parquet")
    db.add_source("labels", url=fixtures.url("tiny_labels.csv"), format="csv")
    classification = db.eval_inference(
        model=fixtures.model("tiny_modernbert_classifier"), source="corpus",
        columns=["content"], task="classification",
        golden_source="labels.public.tiny_labels", label_column="label",
    )
    db.add_source("ner_corpus", url=fixtures.url("tiny_ner_corpus.parquet"), format="parquet")
    db.add_source("ner_gold", url=fixtures.url("tiny_ner_gold.csv"), format="csv")
    ner = db.eval_inference(
        model=fixtures.model("tiny_modernbert_ner"), source="ner_corpus",
        columns=["text"], task="ner",
        golden_source="ner_gold.public.tiny_ner_gold", label_column="label",
    )

    a, b = TENANTS
    with db.tenant_scope(a):
        db.register_channel("scored_by", priority=50,
                            columns=[("score", "Float64"), ("model", "Utf8")])
        db.register_channel("annotated_by", priority=10, columns=[("label", "Utf8")])
        db.add_channel_columns("annotated_by",
                               columns=[("confidence", "Float32"), ("rank", "Int32")])
    with db.tenant_scope(b):
        seen_by_b = db.list_channels()
        db.register_channel("scored_by", priority=3, columns=[("note", "Utf8")])
        b_channels = db.list_channels()
    with db.tenant_scope(a):
        a_channels = db.list_channels()

    return {
        "retrieval": retrieval, "per_query": per_query, "compared": compared,
        "classification": classification, "ner": ner,
        "channels": {"a": a_channels, "b_before": seen_by_b, "b": b_channels,
                     "unbound": db.list_channels()},
    }

## Run it in process, then against a server

`jammi.connect("file://…")` runs the engine in this process;
`jammi.testing.LiveServer` starts a real `jammi-server` (the `jammi-server`
wheel puts one on `PATH`) on ports the kernel assigns, and
`jammi.connect(server.endpoint)` is a `grpc://` session to it. The same
function runs on each.

In [ ]:
with jammi.connect(f"file://{tempfile.mkdtemp()}") as embedded:
    local = evaluate(embedded)

with LiveServer(tempfile.mkdtemp()) as server, jammi.connect(server.endpoint) as remote:
    served = evaluate(remote)
print(f"evaluated in process and at {server.endpoint}")

## The reports

Retrieval quality of the raw embeddings, overall and by cohort. The engine
never interprets a cohort tag; it carries each query's tags on its per-query
record, so a slice is a group-by over those records. The golden asks its first
200 papers by id, so which eras those queries fall in is a property of the
scale's paper set — the slice reports the eras the queries carry:

In [ ]:
from collections import defaultdict

agg = local["retrieval"]["aggregate"]
print("  ".join(f"{m} {agg[m]:.3f}" for m in ("precision_at_k", "recall_at_k", "mrr", "ndcg")))
by_era = defaultdict(list)
for record in local["retrieval"]["per_query"]:
    by_era[record["cohorts"]["era"]].append(record["metrics"]["precision"])
for era, scores in sorted(by_era.items()):
    print(f"  era={era:<5}  queries {len(scores):>3}  precision@10 {sum(scores) / len(scores):.3f}")
print(f"persisted per-query rows: {len(local['per_query'])}")

The comparison: the propagated table's precision\@10 against the raw baseline,
with the paired test's confidence interval on the difference.

In [ ]:
treated = local["compared"]["per_table"][1]
delta = treated["delta"]["precision_at_k"]
significance = treated["delta"]["significance"]["precision_at_k"]
print(f"propagated − raw precision@10: {delta['absolute']:+.3f}  "
      f"(95% CI {significance['ci_lower']:+.3f} … {significance['ci_upper']:+.3f}, "
      f"p = {significance['p_value']:.4f})")

And the inference evaluations. The fixture models are small and randomly
initialised, so their scores are low — what is under test here is the verb, not
the model: a forward pass scored against labels, and against gold entity spans.

In [ ]:
cls, ner = local["classification"]["aggregate"], local["ner"]["aggregate"]
print(f"classification: accuracy {cls['accuracy']:.3f}  f1 {cls['f1']:.3f}")
print(f"ner:            precision {ner['precision']:.3f}  recall {ner['recall']:.3f}  "
      f"f1 {ner['f1']:.3f}")

In [ ]:
for metric in ("precision_at_k", "recall_at_k", "mrr", "ndcg"):
    contracts.assert_close(f"eval.retrieval.{metric}", agg[metric], tol=0.01)
contracts.assert_close("eval.compare.precision_delta", delta["absolute"], tol=0.01)
contracts.assert_close("eval.classification.accuracy", cls["accuracy"])
contracts.assert_close("eval.ner.f1", ner["f1"])
assert len(local["per_query"]) == len(local["retrieval"]["per_query"])

## The same report from either transport

The server's report must equal the in-process one: the same keys at every
level and the same values. Two things legitimately differ and are set aside —
identifiers each engine mints for itself (the run id, the embedding table's
name), and the order of per-query and per-record rows, which the report does not
promise.

In [ ]:
MINTED = {"eval_run_id", "table_name", "embedding_table"}


def canonical(value):
    """A report with minted identifiers dropped and row lists in key order."""
    if isinstance(value, dict):
        return {k: canonical(v) for k, v in value.items() if k not in MINTED}
    if isinstance(value, list):
        rows = [canonical(v) for v in value]
        key = next((k for k in ("query_id", "record_id", "channel_id") if rows and
                    isinstance(rows[0], dict) and k in rows[0]), None)
        return sorted(rows, key=lambda r: r[key]) if key else rows
    return value


def differences(a, b, path=""):
    """Every path at which two canonical reports disagree."""
    if isinstance(a, dict) and isinstance(b, dict):
        return [d for k in a.keys() | b.keys()
                for d in differences(a.get(k), b.get(k), f"{path}.{k}")]
    if isinstance(a, list) and isinstance(b, list) and len(a) == len(b):
        return [d for i, (x, y) in enumerate(zip(a, b)) for d in differences(x, y, f"{path}[{i}]")]
    if isinstance(a, float) and isinstance(b, float):
        return [] if abs(a - b) <= 1e-6 * max(1.0, abs(a), abs(b)) else [path]
    return [] if a == b else [path]


for verb in local:
    print(f"{verb:<15} remote == embedded: {not differences(canonical(local[verb]), canonical(served[verb]))}")

In [ ]:
for verb in local:
    assert not differences(canonical(local[verb]), canonical(served[verb])), verb

## Provenance channels — typed, append-only, tenant-scoped

An **evidence-provenance channel** attaches typed columns of provenance to
records: a channel is an id, a priority, and `(name, dtype)` columns, and its
columns can be appended to but never changed. The registry is scoped per
tenant, and the sequence above read it from each side:

In [ ]:
channels = local["channels"]
annotated = next(c for c in channels["a"] if c["channel_id"] == "annotated_by")
a_scored = next(c for c in channels["a"] if c["channel_id"] == "scored_by")
b_scored = next(c for c in channels["b"] if c["channel_id"] == "scored_by")
leak = {"scored_by", "annotated_by"} & {c["channel_id"] for c in channels["b_before"]}
print(f"annotated_by columns (appended after the original): "
      f"{[c['name'] for c in annotated['columns']]}")
print(f"tenant B saw A's channels: {sorted(leak) or 'none'}")
print(f"scored_by — A: {[c['name'] for c in a_scored['columns']]}  "
      f"B: {[c['name'] for c in b_scored['columns']]}")
print(f"unbound listing: {sorted(c['channel_id'] for c in channels['unbound'])}")

In [ ]:
assert [c["name"] for c in annotated["columns"]] == ["label", "confidence", "rank"]
assert not leak and a_scored["columns"] != b_scored["columns"]
assert {c["channel_id"] for c in channels["unbound"]} == {"vector", "inference", "bm25"}

Tenant B cannot see tenant A's channels, and registers its own `scored_by` —
different columns, no collision with A's. A session bound to no tenant sees only
the global channels every engine seeds, one per way the engine itself produces
evidence: `vector` (similarity search), `inference` (a model's output), and
`bm25` (lexical scoring).

## Bridge note

> **Evaluation is an engine verb, on every transport.** Retrieval quality
> (`eval_embeddings`, sliced by cohort and read back per query), comparison
> (`eval_compare`, with a paired significance test), and inference quality
> (`eval_inference` for classification and NER) return the same nested report
> from an in-process engine and from a server — measured here on every run, not
> asserted once. The provenance channel registry is the typed, append-only,
> per-tenant complement: the engine's isolation applied to the catalog of
> provenance columns.

## References

- Kleppmann, Martin (2017) *Designing Data-Intensive Applications: The Big Ideas Behind Reliable, Scalable, and Maintainable Systems* O'Reilly Media.